In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Define State (Just messages for now)
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# We first define a State for our Graph

class State(TypedDict):
    messages = Annotated[list, add_messages]

In [3]:
# 2. Define Tools and LLM
tools = [TavilySearchResults(max_results=2)]
llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

C:\Users\bpu320145\AppData\Local\Temp\ipykernel_17772\1972188659.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(max_results=2)]


In [4]:
# 3. Define the core Agent Node
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [5]:
# 4. Build the Graph
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

In [6]:
# 5. Define Edges (Routing)
graph_builder.add_edge(START, "chatbot")
# tools_condition automatically routes to "tools" if the LLM calls a tool, otherwise to END
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

# Compile
graph = graph_builder.compile()

In [7]:
# Execute (No UI, just pure logic)
response = graph.invoke({"messages": [("user", "What is the capital of France?")]})
print(response["messages"][-1].content)

The capital of France is Paris.
